I've come to find that the spotipy library is a helpful addon for accessing Spotify's API capabilities in this project.

In [53]:
import spotipy
import pandas as pd
from spotipy.oauth2 import SpotifyClientCredentials
import time
import os
from dotenv import load_dotenv

API Credentials

I did some external research and found that for security reasons, I should try the python-dotenv package.

In [54]:
load_dotenv()

CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET))

data = []

Spotify imposed some changes within their API leading to some frustrations within the data collection process.
I've created a new loop to work with the data available to us.

I'm currently looking to supplement this data with another API to find song key + bpm.

In [55]:
with open('billboard_songs.txt', 'r', encoding='utf-8') as file:
    for line in file:
        if ',' not in line: 
            continue
            
        title, artist = line.strip().split(', ', 1)
        
        primary_artist = artist.split(' & ')[0].split(' Featuring ')[0]
        
        search = sp.search(q=f"track:{title} artist:{primary_artist}", type='track', limit=1)
        
        if search['tracks']['items']:
            track = search['tracks']['items'][0]
            
            row_data = {
                'Title': title, 
                'Artist': artist, 
                'Popularity': track['popularity'],
                'Duration_MS': track['duration_ms'],
                'Explicit': track['explicit'],
                'Release_Date': track['album']['release_date'],
                'Album_Name': track['album']['name']
            }
            
            data.append(row_data)  

        time.sleep(0.2)

In [56]:
pd.DataFrame(data).to_csv('spotify.csv', index=False)